# Filtering & Dashboarding with LangSmith

This notebook sets up your LangSmith project with richly-labeled traces, then guides you through **filtering** and **custom dashboards** in the UI.

Most of this session happens in the browser — this notebook is your reference and exercise sheet.

---

## Part 0 — Setup

`setup_langsmith.py` does everything in the right order:

1. Push agent + eval prompts to LangSmith Hub
2. Create labeled datasets
3. Run **1 trace** → creates the LangSmith project
4. Attach **online evaluators** to the project (automation rules that fire on every new trace)
5. Run all **16 traces** → the evaluators score each one automatically

| Online evaluator | Scores |
|---|---|
| `phishing_detection` | Is this email a phishing attempt? (0/1) |
| `groundedness` | Is the response grounded in the email? (0/1) |
| `email_type` | Category: meeting_request, notification, action_required, … |

In [ ]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(project_root / ".env")

In [ ]:
from utils.setup_langsmith import main
main(force_traces=True)

---

## Part 1 — Filtering

### Why filtering?

With 16+ traces and 3 automated eval scores, you need to quickly find:
- **Failure cases** — traces where an evaluator flagged a problem
- **Specific behaviors** — traces that triggered a particular tool or code path
- **Representative examples** — traces to add to your dataset or annotation queue

The LangSmith filter language is the same whether you're filtering in the UI or via the SDK.

---

### Filter types — reference card

| Goal | UI shortcut | Raw filter |
|---|---|---|
| Only root traces | `Is Trace = true` | `eq(is_root, true)` |
| By tag | `Tag` → select | `has(tags, "email-assistant")` |
| By feedback score | `Feedback Score` → slider | `and(eq(feedback_key, "groundedness"), eq(feedback_score, 0))` |
| By output value | `Output KV` → key/value | `eq(output.classification_decision, "respond")` |
| By metadata | `Metadata` → key/value | `and(eq(metadata_key, "langgraph_node"), eq(metadata_value, "triage_router"))` |
| By latency | `Latency` → range | `gt(latency, "5s")` |
| By run name | `Run Name` → text | `eq(name, "triage_router")` |

> **Tip:** Build a filter in the UI, then click **Copy Filter** to get the raw query string — useful for SDK queries or sharing with teammates.

---

### Trace filter vs. Tree filter

When you're looking at a **child run** (e.g., a single LLM call) but want to filter based on something that's true of the **whole trace** — use a trace filter alongside the regular filter.

| Filter type | What it applies to |
|---|---|
| `filter` | The run itself |
| `trace_filter` | The root trace that the run belongs to |
| `tree_filter` | Any run in the same trace tree |

Example: show only the triage LLM call from traces that later failed groundedness:
```
filter:       and(eq(metadata_key, "langgraph_node"), eq(metadata_value, "triage_router"))
trace_filter: and(eq(feedback_key, "groundedness"), eq(feedback_score, 0))
```

### Filtering exercises

Open your LangSmith project and try each filter. Use the UI filter builder, then verify with the raw query.

**Exercise 1 — Triage decisions**

How many emails were classified as `respond` vs `notify` vs `ignore`?

Filter in the UI:
- `Output KV` → key: `classification_decision`, value: `respond`

Raw query: `eq(output.classification_decision, "respond")`

---

**Exercise 2 — Failed groundedness**

Find traces where the agent's response was *not* grounded in the email.

Raw query: `and(eq(feedback_key, "groundedness"), eq(feedback_score, 0))`

Click into one — what did the agent say that wasn't in the original email?

---

**Exercise 3 — Triage LLM call latency**

The `triage_router` node makes a single LLM call. Find just those spans:

Filter:
- `Run Name` = `triage_router`
- Or: `Metadata` → key: `langgraph_node`, value: `triage_router`

Raw: `and(eq(metadata_key, "langgraph_node"), eq(metadata_value, "triage_router"))`

You can also use `langgraph_step = 1` to find the first LLM call in any trace.

---

**Exercise 4 — Emails that triggered scheduling tools**

Find traces where `schedule_meeting` was called:

- `Run Name` = `schedule_meeting` (matches at the tool step level)
- Use **Tree filter** on root traces: tree filter `eq(name, "schedule_meeting")`

How many emails required calendar scheduling?

---

## Part 2 — Online Evaluators

The three automation rules were created by `setup_langsmith.py`. Confirm they're active:

**In the UI:**  
Project → **Automations** tab → you should see `phishing_detection`, `groundedness`, `email_type`.

Click into a rule to see:
- Which traces it fired on
- The LLM judge's reasoning (in the `comment` field)
- Sampling rate and filter configuration

### Where the scores live

Each evaluator writes a **Feedback** record to the trace. You can see them:
- In the trace detail panel → **Feedback** tab
- In the traces list → hover over the feedback column
- In Charts → as feedback keys to chart over time

---

## Part 3 — Custom Dashboards

Open the **Charts** tab in your project. We'll build four charts.

### Chart 1 — Triage decision breakdown

- **Metric**: Run count
- **Split by**: Output KV → `classification_decision`
- **Filter**: `has(tags, "email-assistant")` + `eq(is_root, true)`

This gives you a bar chart showing how many emails fell into each category.

---

### Chart 2 — Groundedness pass rate

- **Metric**: Feedback score (key = `groundedness`)
- **Chart type**: Line — shows pass rate over time as you add more traces
- **Filter**: `has(tags, "email-assistant")`

---

### Chart 3 — Email type distribution

- **Metric**: Run count
- **Split by**: Feedback value (key = `email_type`)
- **Filter**: `eq(is_root, true)`

---

### Chart 4 — Triage LLM latency

- **Metric**: Latency (p50 and p95)
- **Filter**: `and(eq(metadata_key, "langgraph_node"), eq(metadata_value, "triage_router"))`

This isolates the single LLM call that decides ignore/notify/respond, letting you track its latency independently from the full trace.

---

### Chart 5 — Token usage

- **Metric**: Total tokens (or prompt + completion split)
- **Filter**: `eq(is_root, true)` + `has(tags, "email-assistant")`
- **Split by**: Output KV → `classification_decision`

Do `respond` emails use significantly more tokens than `ignore`?

---

### Key metrics reference

| Metric | What it means |
|---|---|
| `latency_p50` / `latency_p99` | Median and tail latency in seconds |
| `total_tokens` | Prompt + completion tokens combined |
| `total_cost` | Estimated cost in USD |
| `error_rate` | Fraction of traces that errored |
| `feedback_stats` | Aggregated pass rates per feedback key |